# Train `cxr-temporal-multiprior` on Colab

This notebook runs **actual pretraining** on Colab using a MIMIC-CXR
subset zip you upload to Google Drive once.

**What you need:**
1. A T4 (free Colab) or better GPU runtime. A100 strongly recommended
   for `K_max=4` (sequence length = `(K+1)·L = 980` tokens).
2. A single zip file uploaded to your Google Drive containing the
   `p10/, p11/, …, p19/` patient folders — i.e. the standard
   MIMIC-CXR-JPG layout: `pXX/p{subject_id}/s{study_id}/{dicom_id}.jpg`.
   That's it — no other files needed. The CSVs (`subset_out/*.csv`) and
   the upstream BioViL-T weights are already in the repo / auto-
   downloaded by hi-ml respectively.

**Order of operations:**
- Section 1: clone + install + verify GPU
- Section 2: mount Drive, unzip data to /content (local SSD is much
  faster than Drive for the inner training loop)
- Section 3: launch single-GPU training with overridden paths
- Section 4: copy checkpoints back to Drive (optional)

## 1. Setup

In [ ]:
# Clone the repo and enter it
!git clone https://github.com/nprakash1/cxr-temporal-multiprior.git
%cd cxr-temporal-multiprior

In [ ]:
# Install dependencies (~2–3 min)
!pip install -q -r requirements.txt
!pip install -q hi-ml-multimodal

In [ ]:
# GPU check
import torch
assert torch.cuda.is_available(), 'No GPU detected. Runtime → Change runtime type → GPU.'
print('device :', torch.cuda.get_device_name(0))
print('mem    :', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GiB')

## 2. Mount Drive and unzip your MIMIC subset

**Edit `ZIP_PATH` below** to point at your uploaded zip on Drive.
Expected zip layout after extraction:

```
<wherever it lands>/
├── p10/
│   └── p10054639/
│       └── s55344325/
│           └── 2cceb6b7-….jpg
├── p11/
├── …
└── p19/
```

If your zip extracts to `mimic_subset/p10/...`, set
`IMAGE_ROOT = '/content/mimic_subset'`. If it extracts to a `files/`
subdir, point at that instead.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# >>> EDIT THIS to your actual zip path on Drive <<<
ZIP_PATH    = '/content/drive/MyDrive/mimic_subset.zip'
EXTRACT_DIR = '/content'                      # local SSD; fastest for training

import os, time
assert os.path.exists(ZIP_PATH), f'Zip not found at {ZIP_PATH}. Fix ZIP_PATH above.'
print(f'Zip size: {os.path.getsize(ZIP_PATH) / 1024**3:.2f} GiB')

t0 = time.time()
!unzip -q -o "{ZIP_PATH}" -d "{EXTRACT_DIR}"
print(f'Unzip done in {time.time()-t0:.0f}s')

In [ ]:
# Auto-detect the IMAGE_ROOT (the dir containing pXX/ folders).
import os, glob
candidates = []
for root, dirs, _ in os.walk(EXTRACT_DIR):
    # IMAGE_ROOT is the dir whose direct children are pXX/.
    if any(d.startswith('p') and len(d) == 3 and d[1:].isdigit() for d in dirs):
        candidates.append(root)
        break

assert candidates, (
    f'Could not find a pXX/ layout under {EXTRACT_DIR}. '
    f'List the dir manually: !ls {EXTRACT_DIR}'
)
IMAGE_ROOT = candidates[0]
print(f'IMAGE_ROOT = {IMAGE_ROOT}')
print('top-level patient dirs:', sorted(
    d for d in os.listdir(IMAGE_ROOT) if d.startswith('p') and len(d) == 3
)[:15])

In [ ]:
# Quick sanity check: can we resolve a sample image from the train CSV?
import pandas as pd, os
df = pd.read_csv('subset_out/biovilt_pretrain_train_imagelevel.csv')
print(f'rows: {len(df)}  splits: {df["split"].unique()}')
row = df.iloc[0]
pid = str(row['subject_id'])
path = os.path.join(
    IMAGE_ROOT,
    f'p{pid[:2]}', f'p{pid}',
    f's{row["study_id"]}',
    f'{row["dicom_id_curr"]}.jpg',
)
print('sample path:', path)
print('exists?    :', os.path.exists(path))
assert os.path.exists(path), (
    'First sample image not found — your zip layout may differ. '
    'Inspect IMAGE_ROOT manually and update if needed.'
)

## 3. Train

`resume_train.py` reads `IMAGE_ROOT`, `CSV_DIR`, `CHECKPOINT_DIR`,
`LOG_DIR`, and `NUM_WORKERS` from env vars (or equivalent CLI flags),
and uses `--epochs` / `--batch-size` to override defaults for smoke
runs. We launch with `torchrun --nproc_per_node=1` because Colab gives
us a single GPU; the DDP code path still works fine at world_size=1.

**Memory note for K_max:**
- `K_max=1`: ~2L = 392 tokens in the self-attn. Fits easily on T4 (16GB).
- `K_max=4`: 5L = 980 tokens. Needs A100 (40GB+) or batch_size=4 on T4.

Adjust `BATCH_SIZE` below if you OOM.

In [ ]:
# Training config — edit freely
K_MAX      = 4              # 1 = reproduce upstream BioViL-T; 2-4 = multi-prior
BATCH_SIZE = 8              # T4 with K_max=4: try 4; A100 with K_max=4: try 16
EPOCHS     = 5              # smoke run; bump to 50 for full pretraining
MODE       = 'biovilt'      # 'biovil' | 'biovilt' | 'biovilt_finetuned'

import os
os.environ['IMAGE_ROOT']     = IMAGE_ROOT
os.environ['CSV_DIR']        = '/content/cxr-temporal-multiprior/subset_out'
os.environ['CHECKPOINT_DIR'] = '/content/checkpoints'
os.environ['LOG_DIR']        = '/content/logs'
os.environ['NUM_WORKERS']    = '2'   # free Colab only has 2 vCPU

In [ ]:
# The training CSV references both train and val splits via the `split` column,
# but resume_train.py looks for a *_combined_imagelevel.csv for val. The repo
# ships train/val/test separately. Easiest fix: symlink val→combined.
import os, shutil
src = '/content/cxr-temporal-multiprior/subset_out/biovilt_pretrain_val_imagelevel.csv'
dst = '/content/cxr-temporal-multiprior/subset_out/biovilt_pretrain_combined_imagelevel.csv'
if not os.path.exists(dst):
    shutil.copyfile(src, dst)
    print('Created combined CSV (copy of val).')
else:
    print('Combined CSV already exists.')

In [ ]:
# Launch training. torchrun handles the LOCAL_RANK env var that DDP needs.
!cd /content/cxr-temporal-multiprior && \
    torchrun --nproc_per_node=1 --master_port=29501 \
        biovilt/resume_train.py \
        --k-max {K_MAX} \
        --mode  {MODE} \
        --batch-size {BATCH_SIZE} \
        --epochs {EPOCHS}

## 4. Save checkpoints back to Drive (optional)

Colab wipes `/content` between sessions, so persist your trained
weights to Drive if you want to resume later.

In [ ]:
!mkdir -p /content/drive/MyDrive/cxr_temporal_ckpts
!cp /content/checkpoints/best.pt /content/drive/MyDrive/cxr_temporal_ckpts/best_kmax{K_MAX}.pt
!cp /content/logs/val_metrics.csv /content/drive/MyDrive/cxr_temporal_ckpts/val_metrics_kmax{K_MAX}.csv
!ls -lh /content/drive/MyDrive/cxr_temporal_ckpts

## Troubleshooting

**`AssertionError: first sample image not found`** — your zip's
directory structure doesn't have `pXX/` directly under the auto-
detected `IMAGE_ROOT`. Run `!find /content -maxdepth 4 -type d -name 'p1*' | head` to
see the actual layout, then set `IMAGE_ROOT` manually and re-run.

**`CUDA out of memory`** — drop `BATCH_SIZE` to 4 or 2, or use
`K_MAX=2` instead of 4. Free-tier T4 has only 16GB. If still OOM,
Colab Pro's A100 is the cleanest fix.

**`NCCL error`** — should be rare on Colab but if it happens, you can
fall back to gloo by editing the `setup_ddp` call in
`biovilt/resume_train.py` from `nccl` to `gloo`. Single-GPU
performance is identical.

**Training is slow** — the bottleneck is JPEG decoding through 2
DataLoader workers. Set `NUM_WORKERS=4` if you have Colab Pro with
more vCPUs.
